In [1]:
import pandas as pd
import sqlite3
import os



In [2]:
# Load cleaned datasets
funding_rounds = pd.read_csv("funding_rounds_cleaned.csv")
objects = pd.read_csv("objects_cleaned.csv")
investments = pd.read_csv("investments_clean.csv")
funds = pd.read_csv("funds_clean.csv")
acquisitions = pd.read_csv("acquisitions_clean.csv")
ipos = pd.read_csv("ipos_clean.csv")

In [3]:
objects["object_id"] = objects.apply(
    lambda row: "c:" + str(row["entity_id"]) if row["entity_type"] == "Company"
    else "f:" + str(row["entity_id"]) if row["entity_type"] == "FinancialOrg"
    else "p:" + str(row["entity_id"]),
    axis=1
)


In [4]:
conn = sqlite3.connect("venture_capital.db")

In [5]:

funding_rounds.to_sql("funding_rounds", conn, if_exists="replace", index=False)
objects.to_sql("objects", conn, if_exists="replace", index=False)
investments.to_sql("investments", conn, if_exists="replace", index=False)
funds.to_sql("funds", conn, if_exists="replace", index=False)
acquisitions.to_sql("acquisitions", conn, if_exists="replace", index=False)
ipos.to_sql("ipos", conn, if_exists="replace", index=False)

659

# Funding activity over time

In [6]:
query1 = """
SELECT
    strftime('%Y', fr.funded_at) AS funding_year,
    COUNT(fr.id) AS number_of_funding_rounds,
    SUM(fr.raised_amount_usd) AS total_funding_usd,
    AVG(fr.raised_amount_usd) AS avg_funding_usd
FROM funding_rounds fr
LEFT JOIN objects o
    ON fr.object_id = o.object_id
WHERE fr.funded_at IS NOT NULL
  AND fr.raised_amount_usd > 0
GROUP BY funding_year
ORDER BY funding_year;
"""

funding_activity_over_time = pd.read_sql(query1, conn)
funding_activity_over_time.to_csv("funding_activity_over_time.csv", index=False)
funding_activity_over_time

,funding_year,number_of_funding_rounds,total_funding_usd,avg_funding_usd
0,1960,4,5.273603e+07,1.318401e+07
1,1984,2,1.410000e+05,7.050000e+04
2,1987,1,2.500000e+06,2.500000e+06
3,1989,1,1.500000e+04,1.500000e+04
4,1990,1,1.000000e+06,1.000000e+06
5,1992,1,2.000000e+06,2.000000e+06
6,1993,1,1.250000e+05,1.250000e+05
7,1994,2,1.340000e+07,6.700000e+06
8,1995,5,1.730000e+07,3.460000e+06
9,1996,4,1.354250e+07,3.385625e+06


# Top VC funds by capital raised

In [7]:
query2 = """
SELECT
    name AS fund_name,
    funded_at,
    funded_year,
    raised_amount,
    raised_currency_code
FROM funds
WHERE raised_amount > 0
ORDER BY raised_amount DESC
LIMIT 10;
"""

top_vc_funds = pd.read_sql(query2, conn)
top_vc_funds.to_csv("top_vc_funds.csv", index=False)
top_vc_funds

,fund_name,funded_at,funded_year,raised_amount,raised_currency_code
0,Deric R. Mccloud,2013-09-09,2013.0,89000000000,USD
1,Carlyle Partners VI,2013-11-25,2013.0,13000000000,USD
2,Fund VIII,2013-11-07,2013.0,12000000000,USD
3,Warburg Pincus Private Equity XI,2013-05-13,2013.0,11200000000,USD
4,CVC Capital Partners VI fund,2013-07-22,2013.0,10500000000,EUR
5,Silver Lake Partners IV,2013-04-19,2013.0,10300000000,USD
6,Accel Fund,2008-12-01,2008.0,10000000000,USD
7,Venture Fund 2012,2012-04-03,2012.0,10000000000,USD
8,Silver Lake Partners III,2007-01-01,2007.0,9600000000,USD
9,mega-buyout fund,2013-08-08,2013.0,8400000000,USD


# Most active investors

In [8]:
query3 = """
SELECT
    o.name AS investor_name,
    o.entity_type AS investor_type,
    o.category_code,
    COUNT(i.id) AS number_of_investments
FROM investments i
LEFT JOIN objects o
    ON i.investor_object_id = o.object_id
WHERE o.name IS NOT NULL
GROUP BY
    o.name,
    o.entity_type,
    o.category_code
ORDER BY number_of_investments DESC
LIMIT 10;
"""

most_active_investors = pd.read_sql(query3, conn)
most_active_investors.to_csv("most_active_investors.csv", index=False)
most_active_investors

,investor_name,investor_type,category_code,number_of_investments
0,Intel Capital,FinancialOrg,Unknown,529
1,New Enterprise Associates,FinancialOrg,Unknown,515
2,Sequoia Capital,FinancialOrg,Unknown,507
3,SV Angel,FinancialOrg,Unknown,484
4,Accel Partners,FinancialOrg,Unknown,480
5,Kleiner Perkins Caufield & Byers,FinancialOrg,Unknown,478
6,Y Combinator,Company,finance,478
7,Draper Fisher Jurvetson (DFJ),FinancialOrg,Unknown,462
8,500 Startups,FinancialOrg,Unknown,364
9,First Round Capital,FinancialOrg,Unknown,361


# Acquisition vs IPO exit summary

In [9]:
query4 = """
WITH exits AS (
    SELECT
        'Acquisition' AS exit_type,
        acquired_at AS exit_date,
        price_amount AS exit_value
    FROM acquisitions

    UNION ALL

    SELECT
        'IPO' AS exit_type,
        public_at AS exit_date,
        valuation_amount AS exit_value
    FROM ipos
)

SELECT
    exit_type,
    COUNT(*) AS number_of_exits,
    SUM(exit_value) AS total_exit_value,
    AVG(exit_value) AS avg_exit_value
FROM exits
GROUP BY exit_type
ORDER BY number_of_exits DESC;
"""

exit_summary = pd.read_sql(query4, conn)
exit_summary.to_csv("exit_summary.csv", index=False)
exit_summary

,exit_type,number_of_exits,total_exit_value,avg_exit_value
0,Acquisition,9562,3.715975e+12,3.886191e+08
1,IPO,659,4.314230e+11,6.546632e+08


# Largest acquisition and IPO events

In [11]:
query5 = """
WITH exits AS (
    SELECT
        'Acquisition' AS exit_type,
        acquired_object_id AS company_id,
        acquired_at AS exit_date,
        price_amount AS exit_value
    FROM acquisitions

    UNION ALL

    SELECT
        'IPO' AS exit_type,
        object_id AS company_id,
        public_at AS exit_date,
        valuation_amount AS exit_value
    FROM ipos
)

SELECT
    o.name AS company_name,
    o.category_code,
    o.status,
    e.exit_type,
    e.exit_date,
    e.exit_value
FROM exits e
LEFT JOIN objects o
    ON e.company_id = o.object_id
WHERE e.exit_value > 0
ORDER BY e.exit_value DESC
LIMIT 10;
"""

largest_exits = pd.read_sql(query5, conn)

largest_exits.to_csv(
    'largest_exits.csv',
    index=False
)

largest_exits

,company_name,category_code,status,exit_type,exit_date,exit_value
0,None,None,None,Acquisition,1984-06-27,2.600000e+12
1,GREE,games_video,ipo,IPO,2008-12-17,1.089600e+11
2,Facebook,social,ipo,IPO,2012-05-18,1.040000e+11
3,Amazon,ecommerce,ipo,IPO,1997-05-01,1.000000e+11
4,None,None,None,Acquisition,2011-03-20,3.900000e+10
5,None,None,None,Acquisition,2005-08-15,3.500000e+10
6,Alltel,public_relations,acquired,Acquisition,2008-06-05,2.810000e+10
7,None,None,None,Acquisition,2007-07-03,2.600000e+10
8,None,None,None,Acquisition,2011-02-16,2.000000e+10
9,None,None,None,Acquisition,2011-09-22,1.840000e+10
